In [1]:
import os
import sys
import glob
import logging
import json
from pathlib import Path
from datetime import datetime, timedelta
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import rasterio
import torch
import torch.nn.functional as F
from tqdm import tqdm
from collections import defaultdict

# ============================================================
# CONFIGURATION
# ============================================================
class DataConfig:
    SPATIAL_DIMS = (64, 64)
    TEMPORAL_STEPS = 10
    CHANNELS = 15
    
    BAND_NAMES = [
        'SAR_VV', 'SAR_VH', 'Blue', 'Red', 'NIR', 'SWIR', 
        'Temp_2m', 'Precip', 'Max_Temp', 'Min_Temp', 
        'Soil_W1', 'Soil_W3', 'Soil_T1', 'Dewpoint', 'Solar_Rad'
    ]
    
    BAND_INFO = {
        0: {'name': 'SAR_VV', 'range': [-25, 5]},
        1: {'name': 'SAR_VH', 'range': [-30, 0]},
        2: {'name': 'Blue', 'range': [0, 10000]},
        3: {'name': 'Red', 'range': [0, 10000]},
        4: {'name': 'NIR', 'range': [0, 10000]},
        5: {'name': 'SWIR', 'range': [0, 10000]},
        6: {'name': 'Temp_2m', 'range': [250, 320]},      # Kelvin
        7: {'name': 'Precip', 'range': [0, 0.5]},          # Meters (up to 500mm for extreme monsoon events)
        8: {'name': 'Max_Temp', 'range': [250, 320]},      # Kelvin
        9: {'name': 'Min_Temp', 'range': [250, 320]},      # Kelvin (Preserved for BMD Cold Wave ≤16°C / 289.15K)
        10: {'name': 'Soil_W1', 'range': [0, 1.0]},        # m³/m³
        11: {'name': 'Soil_W3', 'range': [0, 1.0]},        # m³/m³
        12: {'name': 'Soil_T1', 'range': [250, 320]},      # Kelvin
        13: {'name': 'Dewpoint', 'range': [250, 320]},     # Kelvin
        14: {'name': 'Solar_Rad', 'range': [0, 25000000]}  # J/m² (up to 25 MJ/m² for extreme clear-sky days)
    }
    
    MAX_NAN_FRACTION = 0.10
    DTYPE_OUTPUT = torch.float32
    MIXED_RESOLUTION = False
    DYNAMIC_ZSCORE = True


# ============================================================
# LOGGING & DIAGNOSTICS
# ============================================================

def setup_logging(output_dir: str) -> logging.Logger:
    """Configure structured logging for pipeline."""
    log_file = os.path.join(output_dir, 'tensor_conversion.log')
    
    logger = logging.getLogger('HazardNetTensorConverter')
    logger.handlers.clear()
    logger.setLevel(logging.INFO)
    logger.propagate = False
    
    fh = logging.FileHandler(log_file, mode='w')
    fh.setLevel(logging.INFO)
    
    ch = logging.StreamHandler()
    ch.setLevel(logging.INFO)
    
    formatter = logging.Formatter('[%(asctime)s] %(levelname)s: %(message)s')
    fh.setFormatter(formatter)
    ch.setFormatter(formatter)
    
    logger.addHandler(fh)
    logger.addHandler(ch)
    
    return logger


# ============================================================
# STAGE 1: INVENTORY & DISCOVERY
# ============================================================

class GeoTIFFInventory:
    """Index all GeoTIFFs by Event_ID and temporal alignment."""
    
    def __init__(self, tiff_dir: str, logger: logging.Logger):
        self.tiff_dir = tiff_dir
        self.logger = logger
        self.inventory: Dict[str, List[str]] = defaultdict(list)
        
    def scan_directory(self) -> Dict[str, List[str]]:
        pattern = os.path.join(self.tiff_dir, '*_T*_15B.tif')
        tiff_files = glob.glob(pattern)
        
        if not tiff_files:
            self.logger.warning(f"No GeoTIFFs found in {self.tiff_dir}")
            return {}
        
        self.logger.info(f"Found {len(tiff_files)} GeoTIFF files")
        
        for tiff_path in tiff_files:
            basename = os.path.basename(tiff_path)
            parts = basename.replace('_15B.tif', '').split('_T')
            if len(parts) == 2:
                event_id = parts[0]
                self.inventory[event_id].append(tiff_path)
        
        self.logger.info(f"Grouped into {len(self.inventory)} unique events")
        return self.inventory
    
    def validate_temporal_alignment(self) -> Tuple[Dict[str, List[str]], Dict[str, str]]:
        valid_inventory = {}
        event_status = {}
        
        for event_id, tiff_paths in self.inventory.items():
            if len(tiff_paths) != DataConfig.TEMPORAL_STEPS:
                event_status[event_id] = f"INCOMPLETE: {len(tiff_paths)}/{DataConfig.TEMPORAL_STEPS} timesteps"
                continue
            
            sorted_paths = sorted(tiff_paths, key=lambda p: int(p.split('_T')[1].split('_')[0]))
            valid_inventory[event_id] = sorted_paths
            event_status[event_id] = "VALID"
        
        n_valid = sum(1 for s in event_status.values() if s == "VALID")
        n_incomplete = len(self.inventory) - n_valid
        self.logger.info(f"Temporal alignment: {n_valid} valid, {n_incomplete} incomplete")
        
        return valid_inventory, event_status


# ============================================================
# STAGE 2: MASK-AWARE SPATIAL VALIDATION
# ============================================================

class TensorValidator:
    """Validate individual GeoTIFF files and apply mask-aware PyTorch resampling."""
    
    def __init__(self, logger: logging.Logger):
        self.logger = logger
        self.resampled_count = 0
    
    def validate_and_process_tiff(self, tiff_path: str) -> Tuple[bool, str, Optional[torch.Tensor], Optional[torch.Tensor]]:
        try:
            with rasterio.open(tiff_path) as src:
                if src.count != DataConfig.CHANNELS:
                    return False, f"Expected {DataConfig.CHANNELS} bands, got {src.count}", None, None
                
                # Read as float32 numpy array (C, H, W)
                img = src.read().astype(np.float32)
                
                # Check NaN fraction
                nan_fraction = np.isnan(img).sum() / img.size
                if nan_fraction > DataConfig.MAX_NAN_FRACTION:
                    return False, f"NaN fraction {nan_fraction:.2%} exceeds limit", None, None
                
                nodata_val = src.nodata if src.nodata is not None else -9999.0
                
                # 1. Convert directly to tensor and generate an explicit, independent boolean mask
                tensor = torch.from_numpy(img)
                valid_mask = ~torch.isnan(tensor) & (tensor != nodata_val)
                
                # 2. Handle Spatial Resampling carefully if sizes mismatch
                if not DataConfig.MIXED_RESOLUTION:
                    h, w = tensor.shape[1], tensor.shape[2]
                    if (h, w) != DataConfig.SPATIAL_DIMS:
                        self.logger.debug(f"Resampling {os.path.basename(tiff_path)} from ({h}, {w}) to {DataConfig.SPATIAL_DIMS}")
                        
                        # Temporarily fill NaNs with 0.0 purely to prevent bilinear NaN explosion
                        tensor_clean = torch.where(valid_mask, tensor, torch.tensor(0.0, dtype=tensor.dtype))
                        
                        # Resample Data (Bilinear) and Mask (Nearest, to preserve strict boundaries)
                        tensor = F.interpolate(
                            tensor_clean.unsqueeze(0), 
                            size=DataConfig.SPATIAL_DIMS, 
                            mode='bilinear', 
                            align_corners=False
                        ).squeeze(0)
                        
                        valid_mask = F.interpolate(
                            valid_mask.unsqueeze(0).float(), 
                            size=DataConfig.SPATIAL_DIMS, 
                            mode='nearest'
                        ).squeeze(0).bool()
                        
                        self.resampled_count += 1
                        
                return True, "OK", tensor, valid_mask
        except Exception as e:
            return False, f"Error reading TIFF: {str(e)}", None, None
    
    def validate_event_tensor(self, tiff_paths: List[str]) -> Tuple[bool, str, Optional[torch.Tensor], Optional[torch.Tensor]]:
        tensors = []
        masks = []
        for t, tiff_path in enumerate(tiff_paths):
            is_valid, msg, tensor, valid_mask = self.validate_and_process_tiff(tiff_path)
            if not is_valid:
                return False, f"Timestep {t}: {msg}", None, None
            tensors.append(tensor)
            masks.append(valid_mask)
        
        # Stack to (T, C, H, W)
        stacked_tensor = torch.stack(tensors, dim=0)
        stacked_mask = torch.stack(masks, dim=0)
        return True, "OK", stacked_tensor, stacked_mask


# ============================================================
# STAGE 3: TRUE DYNAMIC Z-SCORE NORMALIZATION & SERIALIZATION
# ============================================================

class TensorNormalizer:
    """Apply true dynamic z-score normalization per tensor using explicit validity masks."""
    
    def __init__(self, logger: logging.Logger):
        self.logger = logger
        
    def compute_global_statistics(self, valid_raw_tensors: Dict[str, torch.Tensor], valid_masks: Dict[str, torch.Tensor]) -> Dict[str, Dict]:
        """Compute global mean and std per channel across all valid events for reference/logging."""
        self.logger.info("Computing global normalization statistics...")
        stats = {}
        for c, band_name in enumerate(DataConfig.BAND_NAMES):
            all_values = []
            for event_id, tensor in valid_raw_tensors.items():
                mask = valid_masks[event_id]
                # tensor is (T, C, H, W)
                channel_data = tensor[:, c, :, :]
                channel_mask = mask[:, c, :, :]
                
                if channel_mask.sum() > 0:
                    all_values.append(channel_data[channel_mask])
            
            if len(all_values) > 0:
                concatenated = torch.cat(all_values)
                mean = float(concatenated.mean())
                std = float(concatenated.std())
                std = max(std, 1e-6) # Prevent division by zero
            else:
                mean = 0.0
                std = 1.0
                
            stats[band_name] = {'mean': mean, 'std': std}
            self.logger.debug(f"  {band_name}: µ={mean:.4f}, σ={std:.4f}")
            
        return stats
        
    def normalize_tensor(self, tensor: torch.Tensor, valid_mask: torch.Tensor) -> torch.Tensor:
        """
        Applies true Dynamic Z-score normalization per patch and channel.
        Safeguards valid physical zeros and pins invalid regions to a true neutral 0.0.
        """
        normalized = torch.zeros_like(tensor, dtype=DataConfig.DTYPE_OUTPUT)
        
        # Assuming input layout is (T, C, H, W)
        for c in range(DataConfig.CHANNELS):
            channel_data = tensor[:, c, :, :]
            channel_mask = valid_mask[:, c, :, :]
            
            # Calculate statistics strictly using the explicit mask coordinates
            if channel_mask.sum() > 0:
                mu = channel_data[channel_mask].mean()
                sigma = channel_data[channel_mask].std()
                sigma = torch.clamp(sigma, min=1e-6)
                
                # Compute z-score across the whole slice
                z_score = (channel_data - mu) / sigma
                
                # Map valid data to its z-score, and force invalid regions to an absolute 0.0
                normalized[:, c, :, :] = torch.where(channel_mask, z_score, torch.tensor(0.0, dtype=z_score.dtype))
            else:
                # Entire channel slice is invalid padding
                normalized[:, c, :, :] = 0.0
                
        # Permute to (C, T, H, W) for standard PyTorch 3D CNN input
        return normalized.permute(1, 0, 2, 3)


class TensorSerializer:
    """Save normalized PyTorch tensors to disk."""
    
    def __init__(self, output_dir: str, logger: logging.Logger):
        self.output_dir = output_dir
        self.tensor_dir = os.path.join(output_dir, 'tensors')
        self.logger = logger
        os.makedirs(self.tensor_dir, exist_ok=True)
    
    def save_tensor(self, event_id: str, tensor: torch.Tensor) -> str:
        tensor_path = os.path.join(self.tensor_dir, f'{event_id}.pt')
        torch.save(tensor, tensor_path)
        return tensor_path


# ============================================================
# STAGE 4: PIPELINE ORCHESTRATION
# ============================================================

class GeoTIFFTensorPipeline:
    """End-to-end orchestration of GeoTIFF → PyTorch Tensor conversion."""
    
    def __init__(self, tiff_dir: str, csv_path: str, output_dir: str):
        self.tiff_dir = tiff_dir
        self.csv_path = csv_path
        self.output_dir = output_dir
        
        os.makedirs(output_dir, exist_ok=True)
        self.logger = setup_logging(output_dir)
        
        self.inventory = GeoTIFFInventory(tiff_dir, self.logger)
        self.validator = TensorValidator(self.logger)
        self.normalizer = TensorNormalizer(self.logger)
        self.serializer = TensorSerializer(output_dir, self.logger)
        
        self.df_severity = pd.read_csv(csv_path)
        # Convert 'Date' column to datetime objects immediately after loading
        self.df_severity['Date'] = pd.to_datetime(self.df_severity['Date'])
        self.df_severity = self.df_severity[self.df_severity['Hazard_Type'] != 'Earthquake'].copy()
        self.logger.info(f"Filtered out Earthquake events. Remaining events to process: {len(self.df_severity)}")
        
        self.manifest = []
        self.validation_report = defaultdict(list)
    
    def run(self):
        self.logger.info("=" * 70)
        self.logger.info("HazardNet PyTorch GeoTIFF-to-Tensor Pipeline")
        self.logger.info(f"Dynamic Z-Score: {DataConfig.DYNAMIC_ZSCORE}")
        self.logger.info(f"Mask-Aware Validation: Enabled")
        self.logger.info("=" * 70)
        
        # Stage 1: Inventory
        self.logger.info("\n[STAGE 1] Scanning directory and validating temporal alignment...")
        self.inventory.scan_directory()
        valid_inventory, event_status = self.inventory.validate_temporal_alignment()
        
        # Stage 2: Mask-Aware Validation and Resampling
        self.logger.info("\n[STAGE 2] Validating and Resampling GeoTIFFs (Mask-Aware)...")
        valid_raw_events = {}
        valid_masks_events = {}
        target_event_ids = set(self.df_severity['Event_ID_Internal'].values)
        
        for event_id, tiff_paths in tqdm(valid_inventory.items(), desc="Validate Events"):
            if event_id not in target_event_ids:
                self.logger.debug(f"Skipping {event_id}: Filtered out by hazard type.")
                continue
            is_valid, msg, tensor_stacked, mask_stacked = self.validator.validate_event_tensor(tiff_paths)
            if is_valid:
                valid_raw_events[event_id] = tensor_stacked
                valid_masks_events[event_id] = mask_stacked
                self.validation_report[event_id].append(('VALID', msg))
            else:
                self.validation_report[event_id].append(('INVALID', msg))
                self.logger.warning(f"{event_id}: {msg}")
        
        n_valid = len(valid_raw_events)
        n_total = len(valid_inventory)
        self.logger.info(f"Passed validation: {n_valid}/{n_total} events ({100*n_valid/n_total:.1f}%)")
        self.logger.info(f"Resampled {self.validator.resampled_count} GeoTIFFs to standard dimensions")
        
        # Stage 3: Compute and save global normalization statistics
        self.logger.info("\n[STAGE 3] Computing and saving global normalization statistics...")
        global_stats = self.normalizer.compute_global_statistics(valid_raw_events, valid_masks_events)
        stats_path = os.path.join(self.output_dir, 'normalization_stats.json')
        with open(stats_path, 'w') as f:
            json.dump(global_stats, f, indent=2)
        self.logger.info(f"Global normalization stats saved to {stats_path}")
        
        # Stage 4: Normalization & Serialization
        self.logger.info("\n[STAGE 4] Normalizing and Serializing PyTorch tensors...")
        for event_id, tensor_stacked in tqdm(valid_raw_events.items(), desc="Normalize & Serialize"):
            try:
                mask_stacked = valid_masks_events[event_id]
                
                # Normalize dynamically using explicit mask
                tensor_normalized = self.normalizer.normalize_tensor(tensor_stacked, mask_stacked)
                tensor_path = self.serializer.save_tensor(event_id, tensor_normalized) # Get absolute path
                
                # Link to severity metadata
                severity_row = self.df_severity[self.df_severity['Event_ID_Internal'] == event_id]
                if not severity_row.empty:
                    date = (severity_row.iloc[0]['Date']).strftime('%Y-%m-%d')
                    district = severity_row.iloc[0]['District']
                    latitude = float(severity_row.iloc[0]['Latitude'])
                    longitude = float(severity_row.iloc[0]['Longitude'])
                    hazard_type = severity_row.iloc[0]['Hazard_Type']
                    severity_idx = float(severity_row.iloc[0]['Severity_Index'])
                    confidence = float(severity_row.iloc[0]['Confidence'])
                else:
                    date = ''
                    district = ''
                    latitude = 0.0
                    longitude = 0.0
                    hazard_type, severity_idx, confidence = 'Unknown', 0.0, 0.5
                
                self.manifest.append({
                    'event_id': event_id,
                    'date': date,
                    'district': district,
                    'latitude': latitude,
                    'longitude': longitude,
                    'tensor_path': tensor_path, # Store relative path
                    'shape': f"C={tensor_normalized.shape[0]}, T={tensor_normalized.shape[1]}, H={tensor_normalized.shape[2]}, W={tensor_normalized.shape[3]}",
                    'hazard_type': hazard_type,
                    'severity_index': severity_idx,
                    'confidence': confidence,
                    'status': 'OK'
                })
            except Exception as e:
                self.logger.error(f"{event_id}: Serialization failed: {str(e)}")
                self.manifest.append({
                    'event_id': event_id,
                    'date': '',
                    'district': '',
                    'latitude': 0.0,
                    'longitude': 0.0,
                    'tensor_path': '',
                    'shape': '',
                    'hazard_type': '',
                    'severity_index': 0.0,
                    'confidence': 0.5,
                    'status': f'ERROR: {str(e)}'
                })
        
        # Stage 5: Export manifest
        self.logger.info("\n[STAGE 5] Exporting manifest...")
        df_manifest = pd.DataFrame(self.manifest)
        manifest_path = os.path.join(self.output_dir, 'tensor_manifest.csv')
        df_manifest.to_csv(manifest_path, index=False)
        self.logger.info(f"Manifest saved to {manifest_path}")
        
        # Export validation report
        report_path = os.path.join(self.output_dir, 'tensor_validation_report.txt')
        with open(report_path, 'w') as f:
            f.write("HazardNet Tensor Validation Report\n")
            f.write("=" * 70 + "\n\n")
            for event_id, messages in self.validation_report.items():
                f.write(f"{event_id}:\n")
                for status, msg in messages:
                    f.write(f"  [{status}] {msg}\n")
            f.write(f"\n\nSummary:\n")
            f.write(f"  Total events: {len(self.validation_report)}\n")
            f.write(f"  Valid: {n_valid}\n")
            f.write(f"  Passed: {df_manifest['status'].eq('OK').sum()}\n")
            f.write(f"  Resampled: {self.validator.resampled_count}\n")
        
        self.logger.info(f"Validation report saved to {report_path}")
        self.logger.info("\n" + "=" * 70)
        self.logger.info("PIPELINE COMPLETE")
        self.logger.info("=" * 70)
        
        return df_manifest

if __name__ == '__main__':
    TIFF_DIR = '/kaggle/input/datasets/ashifahmedshuvo/hazardnet/Tensors'
    CSV_PATH = '/kaggle/input/datasets/ashifahmedshuvo/hazardnet/Events/hazardnet_with_severity.csv'
    OUTPUT_DIR = 'tensors_output'
    
    pipeline = GeoTIFFTensorPipeline(TIFF_DIR, CSV_PATH, OUTPUT_DIR)
    df_manifest = pipeline.run()
    
    print("\n" + "=" * 70)
    print("TENSOR MANIFEST SUMMARY")
    print("=" * 70)
    print(df_manifest.groupby(['hazard_type', 'status']).size().unstack(fill_value=0))

[2026-07-25 05:13:20,219] INFO: Filtered out Earthquake events. Remaining events to process: 2931
[2026-07-25 05:13:20,220] INFO: ======================================================================
[2026-07-25 05:13:20,221] INFO: HazardNet PyTorch GeoTIFF-to-Tensor Pipeline
[2026-07-25 05:13:20,222] INFO: Dynamic Z-Score: True
[2026-07-25 05:13:20,222] INFO: Mask-Aware Validation: Enabled
[2026-07-25 05:13:20,223] INFO: ======================================================================
[2026-07-25 05:13:20,224] INFO: 
[STAGE 1] Scanning directory and validating temporal alignment...
[2026-07-25 05:13:20,509] INFO: Found 30590 GeoTIFF files
[2026-07-25 05:13:20,531] INFO: Grouped into 3059 unique events
[2026-07-25 05:13:20,546] INFO: Temporal alignment: 3059 valid, 0 incomplete
[2026-07-25 05:13:20,547] INFO: 
[STAGE 2] Validating and Resampling GeoTIFFs (Mask-Aware)...
Validate Events: 100%|██████████| 3059/3059 [09:43<00:00,  5.24it/s]
[2026-07-25 05:23:04,313] INFO: Passed va


TENSOR MANIFEST SUMMARY
status               OK
hazard_type            
Cold Wave           322
Drought              64
Fire                 65
Flash Flood         331
Flood               890
Heat Wave            36
Severe Local Storm  514
Tropical Cyclone    709


In [2]:
"""
================================================================================
HazardNet Unified Experimental Dataset Builder
================================================================================

VALIDATION STRATEGIES:
  0. Event-Based 5-Fold Stratified CV — Baseline performance (hazard-stratified)
  1. Spatial LODO (Division-Level, 8 folds) — Climatologically coherent spatial units
  2. Temporal Split (Season-Adaptive Boundaries) — Independent splits per cropping season
     Kharif-I/II: Train 2000-2017 | Val 2018-2021 | Test 2022-2025
     Rabi:        Train 2000-2020 | Val 2021-2023 | Test 2024-2025
  3. Combined Spatio-Temporal (Division × Season × Era) — Up to 24 folds

MASTER HDF5 ARCHITECTURE:
  - Single master_tensors.h5 built once (~2.5 GB)
  - Lightweight CSV fold manifests reference event_ids from master
  - Worker-safe MasterHDF5Dataset with augmentation + spatial resize

REFERENCES:
  - Islam et al. (2020): Three cropping seasons (Kharif-I, Kharif-II, Rabi)
  - Rahman et al. (2017): Climate non-stationarity in Bangladesh droughts
  - BARC (2023): Bangladesh administrative divisions as agro-climatic zones

================================================================================
"""

import os
import json
import logging
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, get_worker_info
from sklearn.model_selection import StratifiedKFold, train_test_split
import h5py
from tqdm import tqdm
from typing import Dict, List, Tuple, Optional
from pathlib import Path
from collections import defaultdict


# ============================================================================
# CONFIGURATION
# ============================================================================

class DataConfig:
    """Dataset and experimental validation configuration."""
    SEED = 42
    N_FOLDS = 5  # For event-based stratified CV

    # THREE-SEASON CROPPING CALENDAR (Islam et al., 2020)
    CROPPING_SEASONS = {
        'Kharif_I': [3, 4, 5],           # Mar-May
        'Kharif_II': [6, 7, 8, 9, 10],   # Jun-Oct
        'Rabi': [11, 12, 1, 2]           # Nov-Feb
    }

    # Season-specific climate signatures (Islam et al., 2020)
    SEASON_CLIMATE_SIGNATURES = {
        'Kharif_I': {'temp_humidity_corr': -0.60, 'primary_driver': 'temperature'},
        'Kharif_II': {'rainfall_humidity_corr': None, 'primary_driver': 'rainfall'},
        'Rabi': {'rainfall_humidity_corr': 0.75, 'primary_driver': 'rainfall+humidity'}
    }

    # Season-adaptive temporal split boundaries
    SEASON_TEMPORAL_SPLITS = {
        'Kharif_I': {
            'train_end': 2017,
            'val_end': 2021,
            'rationale': 'High density; standard 3-era split aligned with Sentinel-2B launch'
        },
        'Kharif_II': {
            'train_end': 2017,
            'val_end': 2021,
            'rationale': 'Highest density; standard 3-era split aligned with Sentinel-2B launch'
        },
        'Rabi': {
            'train_end': 2020,
            'val_end': 2023,
            'rationale': 'Low density; extended train to preserve test samples'
        }
    }

    # District-to-Division mapping for spatial validation
    DISTRICT_TO_DIVISION = {
        # Dhaka Division (13 districts)
        'Dhaka': 'Dhaka', 'Gazipur': 'Dhaka', 'Narayanganj': 'Dhaka',
        'Tangail': 'Dhaka', 'Manikganj': 'Dhaka', 'Munshiganj': 'Dhaka',
        'Rajbari': 'Dhaka', 'Faridpur': 'Dhaka', 'Gopalganj': 'Dhaka',
        'Madaripur': 'Dhaka', 'Shariatpur': 'Dhaka', 'Kishoreganj': 'Dhaka',
        'Narsingdi': 'Dhaka',
        # Chittagong Division (11 districts)
        'Chattogram': 'Chittagong', 'Cox\'s Bazar': 'Chittagong',
        'Feni': 'Chittagong', 'Noakhali': 'Chittagong', 'Lakshmipur': 'Chittagong',
        'Chandpur': 'Chittagong', 'Brahmanbaria': 'Chittagong',
        'Comilla': 'Chittagong', 'Cumilla': 'Chittagong',
        'Khagrachari': 'Chittagong', 'Rangamati': 'Chittagong', 'Bandarban': 'Chittagong',
        # Rajshahi Division (8 districts)
        'Rajshahi': 'Rajshahi', 'Bogra': 'Rajshahi', 'Joypurhat': 'Rajshahi',
        'Naogaon': 'Rajshahi', 'Natore': 'Rajshahi', 'Chapainawabganj': 'Rajshahi',
        'Pabna': 'Rajshahi', 'Sirajganj': 'Rajshahi',
        # Khulna Division (10 districts)
        'Khulna': 'Khulna', 'Bagerhat': 'Khulna', 'Chuadanga': 'Khulna',
        'Jessore': 'Khulna', 'Jhenaidah': 'Khulna', 'Kushtia': 'Khulna',
        'Magura': 'Khulna', 'Meherpur': 'Khulna', 'Narail': 'Khulna',
        'Satkhira': 'Khulna',
        # Barisal Division (6 districts)
        'Barishal': 'Barisal', 'Barguna': 'Barisal', 'Bhola': 'Barisal',
        'Jhalokati': 'Barisal', 'Patuakhali': 'Barisal', 'Pirojpur': 'Barisal',
        # Sylhet Division (4 districts)
        'Sylhet': 'Sylhet', 'Habiganj': 'Sylhet', 'Moulvibazar': 'Sylhet',
        'Sunamganj': 'Sylhet',
        # Rangpur Division (8 districts)
        'Rangpur': 'Rangpur', 'Dinajpur': 'Rangpur', 'Gaibandha': 'Rangpur',
        'Kurigram': 'Rangpur', 'Lalmonirhat': 'Rangpur', 'Nilphamari': 'Rangpur',
        'Panchagarh': 'Rangpur', 'Thakurgaon': 'Rangpur',
        # Mymensingh Division (4 districts)
        'Mymensingh': 'Mymensingh', 'Jamalpur': 'Mymensingh',
        'Netrokona': 'Mymensingh', 'Sherpur': 'Mymensingh',
    }

    MIN_TEST_EVENTS = 5
    TARGET_TENSOR_SHAPE = (15, 10, 64, 64)
    APPLY_DYNAMIC_ZSCORE = False

    AUGMENT_TRAIN = True
    AUGMENT_PARAMS = {
        'brightness': 0.1,
        'contrast': 0.1,
        'temporal_shift': 1
    }


# ============================================================================
# LOGGING
# ============================================================================

def setup_logger(output_dir: str) -> logging.Logger:
    logger = logging.getLogger('HazardNetExperimentalBuilder')
    logger.handlers.clear()
    logger.setLevel(logging.INFO)
    logger.propagate = False

    fh = logging.FileHandler(os.path.join(output_dir, 'experimental_builder.log'), mode='w')
    fh.setLevel(logging.INFO)
    formatter = logging.Formatter('[%(asctime)s] %(levelname)s: %(message)s')
    fh.setFormatter(formatter)
    logger.addHandler(fh)

    ch = logging.StreamHandler()
    ch.setLevel(logging.INFO)
    ch.setFormatter(formatter)
    logger.addHandler(ch)

    return logger


# ============================================================================
# HAZARD TYPE ENCODING
# ============================================================================

class HazardEncoder:
    HAZARD_TYPES = [
        'Cold Wave', 'Drought', 'Fire', 'Flash Flood',
        'Flood', 'Heat Wave', 'Severe Local Storm', 'Tropical Cyclone'
    ]

    HAZARD_TO_IDX = {h: i for i, h in enumerate(HAZARD_TYPES)}
    IDX_TO_HAZARD = {i: h for i, h in enumerate(HAZARD_TYPES)}

    @classmethod
    def encode(cls, hazard: str) -> int:
        return cls.HAZARD_TO_IDX.get(hazard, -1)

    @classmethod
    def decode(cls, idx: int) -> str:
        return cls.IDX_TO_HAZARD.get(idx, 'Unknown')

    @classmethod
    def n_classes(cls) -> int:
        return len(cls.HAZARD_TYPES)


# ============================================================================
# UTILITY: Season Assignment
# ============================================================================

def assign_cropping_season(month: int) -> str:
    """Assign cropping season per Islam et al. (2020)."""
    if month in DataConfig.CROPPING_SEASONS['Kharif_I']:
        return 'Kharif_I'
    elif month in DataConfig.CROPPING_SEASONS['Kharif_II']:
        return 'Kharif_II'
    elif month in DataConfig.CROPPING_SEASONS['Rabi']:
        return 'Rabi'
    else:
        raise ValueError(f"Invalid month: {month}")


# ============================================================================
# MASTER HDF5 BUILDER (Built ONCE to save disk space)
# ============================================================================

class MasterHDF5Serializer:
    """Builds a single Master HDF5 file containing all unique event tensors."""

    def __init__(self, tensor_dir: str, output_path: str, logger: logging.Logger):
        self.tensor_dir = tensor_dir
        self.output_path = output_path
        self.logger = logger

    def build_master_hdf5(self, df_manifest: pd.DataFrame) -> set:
        self.logger.info(f"Building Master HDF5 at {self.output_path}...")

        unique_events = df_manifest.drop_duplicates(subset=['event_id']).copy()
        successful_event_ids = set()
        failed_count = 0

        with h5py.File(self.output_path, 'w') as h5f:
            grp_tensors = h5f.create_group('tensors')
            grp_labels = h5f.create_group('labels')
            grp_severity = h5f.create_group('severity')
            grp_confidence = h5f.create_group('confidence')

            for _, row in tqdm(unique_events.iterrows(), total=len(unique_events), desc="Building Master HDF5"):
                event_id = row['event_id']
                event_id_str = str(event_id)
                raw_path = str(row['tensor_path'])
                hazard_type = row['hazard_type']

                severity_idx = float(row.get('severity_index', 0.0))
                confidence = float(row.get('confidence', 0.5))

                # Dynamic Path Resolution
                if os.path.exists(raw_path):
                    tensor_path = raw_path
                else:
                    filename = os.path.basename(raw_path)
                    tensor_path = os.path.join(self.tensor_dir, filename)

                if not os.path.exists(tensor_path):
                    self.logger.warning(f"Tensor not found: {tensor_path}")
                    failed_count += 1
                    continue

                try:
                    tensor = torch.load(tensor_path, map_location='cpu')
                    class_idx = HazardEncoder.encode(hazard_type)

                    if class_idx == -1:
                        self.logger.warning(f"Unknown hazard type for {event_id_str}: {hazard_type}")
                        failed_count += 1
                        continue

                    grp_tensors.create_dataset(
                        event_id_str, data=tensor.numpy(),
                        compression='gzip', compression_opts=4
                    )
                    grp_labels.create_dataset(event_id_str, data=class_idx)
                    grp_severity.create_dataset(event_id_str, data=severity_idx)
                    grp_confidence.create_dataset(event_id_str, data=confidence)

                    successful_event_ids.add(event_id_str)

                except Exception as e:
                    self.logger.error(f"Failed to serialize {event_id_str}: {e}")
                    failed_count += 1

            h5f.attrs['n_events'] = len(successful_event_ids)
            h5f.attrs['seed'] = DataConfig.SEED
            h5f.attrs['n_classes'] = HazardEncoder.n_classes()
            h5f.attrs['target_tensor_shape'] = DataConfig.TARGET_TENSOR_SHAPE

        self.logger.info(f"Master HDF5 built successfully: {len(successful_event_ids)} events serialized.")
        return successful_event_ids


# ============================================================================
# STRATEGY 0: EVENT-BASED 5-FOLD STRATIFIED CV
#  NEWLY INTEGRATED: Baseline performance evaluation
# ============================================================================

class EventBasedKFoldBuilder:
    """
    Standard 5-fold stratified cross-validation by hazard type.
    Serves as the baseline performance metric (IEEE TGRS Table II).
    """

    def __init__(self, df_manifest: pd.DataFrame, logger: logging.Logger):
        self.df = df_manifest.copy()
        self.logger = logger
        self.df = self.df[self.df['hazard_type'] != 'Earthquake'].copy()
        self.df['hazard_idx'] = self.df['hazard_type'].apply(HazardEncoder.encode)
        self.df = self.df[self.df['hazard_idx'] >= 0].copy()

        self.logger.info(f"  Event-Based K-Fold:")
        self.logger.info(f"    Total events: {len(self.df)}")
        self.logger.info(f"    Hazard distribution:\n{self.df['hazard_type'].value_counts().to_string()}")

    def partition(self) -> List[Dict]:
        skf = StratifiedKFold(
            n_splits=DataConfig.N_FOLDS,
            shuffle=True,
            random_state=DataConfig.SEED
        )

        folds = []
        X = self.df[['event_id']].values
        y = self.df['hazard_idx'].values

        for fold_idx, (train_val_idx, test_idx) in enumerate(skf.split(X, y)):
            train_val_data = self.df.iloc[train_val_idx].copy()
            test_data = self.df.iloc[test_idx].copy()

            val_ratio = 0.15 / (0.70 + 0.15)
            train_data, val_data = train_test_split(
                train_val_data,
                test_size=val_ratio,
                stratify=train_val_data['hazard_idx'],
                random_state=DataConfig.SEED + fold_idx
            )

            folds.append({
                'fold_type': 'event_kfold',
                'fold_idx': fold_idx,
                'n_events': len(test_data),
                'train': train_data.reset_index(drop=True),
                'val': val_data.reset_index(drop=True),
                'test': test_data.reset_index(drop=True)
            })

            self.logger.info(f"    Fold {fold_idx}: train={len(train_data)}, "
                             f"val={len(val_data)}, test={len(test_data)}")

        self.logger.info(f"  Total Event-Based K-Fold folds: {len(folds)}")
        return folds


# ============================================================================
# STRATEGY 1: SPATIAL LODO (Division-Level)
# ============================================================================

class SpatialLODOBuilder:
    """Leave-One-Division-Out spatial cross-validation (8 climatologically coherent units)."""

    def __init__(self, df_manifest: pd.DataFrame, logger: logging.Logger):
        self.df = df_manifest.copy()
        self.logger = logger
        self.df = self.df[self.df['hazard_type'] != 'Earthquake'].copy()
        self.df['hazard_idx'] = self.df['hazard_type'].apply(HazardEncoder.encode)
        self.df = self.df[self.df['hazard_idx'] >= 0].copy()

        self.df['division'] = self.df['district'].map(DataConfig.DISTRICT_TO_DIVISION)
        unmapped = self.df[self.df['division'].isna()]['district'].unique()
        if len(unmapped) > 0:
            self.logger.warning(f"  Unmapped districts (excluded from spatial LODO): {unmapped.tolist()}")
            self.df = self.df.dropna(subset=['division'])

        self.division_counts = self.df['division'].value_counts()
        self.divisions = sorted(self.df['division'].unique())

        self.logger.info(f"  Spatial LODO (Division-Level):")
        self.logger.info(f"    Divisions: {len(self.divisions)}")
        for div in self.divisions:
            count = self.division_counts.get(div, 0)
            self.logger.info(f"      {div}: {count} events")

    def partition(self) -> List[Dict]:
        folds = []

        for division in self.divisions:
            test_mask = self.df['division'] == division
            train_all = self.df[~test_mask].copy()
            test_data = self.df[test_mask].copy()

            val_ratio = 0.15 / (0.70 + 0.15)
            hazard_counts = train_all['hazard_idx'].value_counts()
            can_stratify = hazard_counts.min() >= 2 if not hazard_counts.empty else False

            train_data, val_data = train_test_split(
                train_all, test_size=val_ratio,
                stratify=train_all['hazard_idx'] if can_stratify else None,
                random_state=DataConfig.SEED
            )

            folds.append({
                'fold_type': 'lodo_division',
                'division': division,
                'n_events': len(test_data),
                'train': train_data.reset_index(drop=True),
                'val': val_data.reset_index(drop=True),
                'test': test_data.reset_index(drop=True)
            })

        self.logger.info(f"  Total Spatial LODO folds: {len(folds)}")
        return folds


# ============================================================================
# STRATEGY 2: TEMPORAL SPLIT (Season-Adaptive Boundaries)
# ============================================================================

class TemporalSplitBuilder:
    """Season-Adaptive Temporal Cross-Validation with independent boundaries per season."""

    def __init__(self, df_manifest: pd.DataFrame, logger: logging.Logger):
        self.df = df_manifest.copy()
        self.logger = logger
        self.df = self.df[self.df['hazard_type'] != 'Earthquake'].copy()
        self.df['hazard_idx'] = self.df['hazard_type'].apply(HazardEncoder.encode)
        self.df = self.df[self.df['hazard_idx'] >= 0].copy()

        self.df['year'] = pd.to_datetime(self.df['date']).dt.year
        self.df['month'] = pd.to_datetime(self.df['date']).dt.month
        self.df['cropping_season'] = self.df['month'].apply(assign_cropping_season)

    def partition(self) -> Dict[str, pd.DataFrame]:
        all_train, all_val, all_test = [], [], []

        self.logger.info(f"  Temporal Split (Season-Adaptive Boundaries):")

        for season, bounds in DataConfig.SEASON_TEMPORAL_SPLITS.items():
            season_df = self.df[self.df['cropping_season'] == season].copy()
            train_end = bounds['train_end']
            val_end = bounds['val_end']

            train_data = season_df[season_df['year'] <= train_end]
            val_data = season_df[(season_df['year'] > train_end) & (season_df['year'] <= val_end)]
            test_data = season_df[season_df['year'] > val_end]

            all_train.append(train_data)
            all_val.append(val_data)
            all_test.append(test_data)

            self.logger.info(f"    {season} ({bounds['rationale']}):")
            self.logger.info(f"      Train (2000-{train_end}): {len(train_data)} events")
            self.logger.info(f"      Val ({train_end+1}-{val_end}):   {len(val_data)} events")
            self.logger.info(f"      Test ({val_end+1}-2025):  {len(test_data)} events")

        combined_train = pd.concat(all_train, ignore_index=True)
        combined_val = pd.concat(all_val, ignore_index=True)
        combined_test = pd.concat(all_test, ignore_index=True)

        self.logger.info(f"\n  Combined Totals:")
        self.logger.info(f"    Train: {len(combined_train)} events")
        self.logger.info(f"    Val:   {len(combined_val)} events")
        self.logger.info(f"    Test:  {len(combined_test)} events")

        return {
            'fold_type': 'temporal_season_adaptive',
            'train': combined_train.reset_index(drop=True),
            'val': combined_val.reset_index(drop=True),
            'test': combined_test.reset_index(drop=True),
            'season_boundaries': DataConfig.SEASON_TEMPORAL_SPLITS
        }


# ============================================================================
# STRATEGY 3: COMBINED SPATIO-TEMPORAL (Division × Season × Era)
# ============================================================================

class SpatioTemporalBuilder:
    """Combined Spatio-Temporal validation at DIVISION level with season-adaptive eras."""

    def __init__(self, df_manifest: pd.DataFrame, logger: logging.Logger):
        self.df = df_manifest.copy()
        self.logger = logger
        self.df = self.df[self.df['hazard_type'] != 'Earthquake'].copy()
        self.df['hazard_idx'] = self.df['hazard_type'].apply(HazardEncoder.encode)
        self.df = self.df[self.df['hazard_idx'] >= 0].copy()

        self.df['year'] = pd.to_datetime(self.df['date']).dt.year
        self.df['month'] = pd.to_datetime(self.df['date']).dt.month
        self.df['cropping_season'] = self.df['month'].apply(assign_cropping_season)

        self.df['division'] = self.df['district'].map(DataConfig.DISTRICT_TO_DIVISION)
        unmapped = self.df[self.df['division'].isna()]['district'].unique()
        if len(unmapped) > 0:
            self.logger.warning(f"  Unmapped districts (excluded from ST folds): {unmapped.tolist()}")
            self.df = self.df.dropna(subset=['division'])

    def partition(self) -> List[Dict]:
        folds = []
        seasons = ['Kharif_I', 'Kharif_II', 'Rabi']
        divisions = sorted(self.df['division'].unique())

        self.logger.info(f"  Spatio-Temporal (Division-Level × Season-Adaptive): "
                         f"{len(divisions)} divisions × {len(seasons)} seasons")

        skipped = 0
        for division in divisions:
            for season in seasons:
                bounds = DataConfig.SEASON_TEMPORAL_SPLITS[season]
                train_end = bounds['train_end']
                val_end = bounds['val_end']

                test_mask = (self.df['division'] == division) & \
                            (self.df['cropping_season'] == season) & \
                            (self.df['year'] > val_end)
                test_data = self.df[test_mask].copy()

                if len(test_data) < DataConfig.MIN_TEST_EVENTS:
                    skipped += 1
                    continue

                train_mask = (self.df['division'] != division) & \
                             (self.df['cropping_season'] == season) & \
                             (self.df['year'] <= train_end)
                train_data = self.df[train_mask].copy()

                val_mask = (self.df['division'] != division) & \
                           (self.df['cropping_season'] == season) & \
                           (self.df['year'] > train_end) & \
                           (self.df['year'] <= val_end)
                val_data = self.df[val_mask].copy()

                if len(train_data) == 0 or len(val_data) == 0:
                    skipped += 1
                    continue

                folds.append({
                    'fold_type': 'spatio_temporal_division',
                    'division': division,
                    'season': season,
                    'n_test_events': len(test_data),
                    'train': train_data.reset_index(drop=True),
                    'val': val_data.reset_index(drop=True),
                    'test': test_data.reset_index(drop=True)
                })

        self.logger.info(f"  Total Spatio-Temporal folds: {len(folds)} "
                         f"(skipped {skipped} with <{DataConfig.MIN_TEST_EVENTS} test events)")
        return folds


# ============================================================================
# ORCHESTRATOR
# ============================================================================

class ExperimentalDatasetBuilder:
    """Builds single Master HDF5 file once and exports CSV fold manifests for ALL strategies."""

    REQUIRED_COLUMNS = ['event_id', 'hazard_type', 'tensor_path', 'severity_index', 'confidence', 'district', 'date']

    def __init__(self, df_manifest_path: str, tensor_dir: str, output_dir: str):
        self.df_manifest_path = df_manifest_path
        self.tensor_dir = tensor_dir
        self.output_dir = output_dir

        os.makedirs(output_dir, exist_ok=True)
        self.logger = setup_logger(output_dir)

        self.df_manifest = pd.read_csv(df_manifest_path)
        self.logger.info(f"Loaded manifest: {len(self.df_manifest)} events")

        missing_cols = [c for c in self.REQUIRED_COLUMNS if c not in self.df_manifest.columns]
        if missing_cols:
            raise ValueError(f"Manifest missing required columns: {missing_cols}. "
                             f"Ensure manifest contains: {self.REQUIRED_COLUMNS}")

        # Compute class weights
        df_filtered = self.df_manifest[self.df_manifest['hazard_type'] != 'Earthquake'].copy()
        df_filtered['hazard_idx'] = df_filtered['hazard_type'].apply(HazardEncoder.encode)
        df_filtered = df_filtered[df_filtered['hazard_idx'] >= 0]

        value_counts = df_filtered['hazard_idx'].value_counts()
        total = len(df_filtered)
        self.class_weights = {}
        for class_idx, count in value_counts.items():
            self.class_weights[class_idx] = total / (len(value_counts) * count)

    def _build_fold_directory(self, fold_data: Dict, fold_dir: str):
        """Saves lightweight CSV fold manifests with column validation."""
        os.makedirs(fold_dir, exist_ok=True)
        for split_name in ['train', 'val', 'test']:
            split_df = fold_data[split_name]
            if 'hazard_idx' not in split_df.columns and 'hazard_type' in split_df.columns:
                split_df = split_df.copy()
                split_df['hazard_idx'] = split_df['hazard_type'].apply(HazardEncoder.encode)
            split_df.to_csv(os.path.join(fold_dir, f'{split_name}_events.csv'), index=False)

    def build_event_kfold(self):
        """ NEW: Strategy 0 — Event-Based 5-Fold Stratified CV."""
        self.logger.info("\n" + "=" * 70)
        self.logger.info("STRATEGY 0: EVENT-BASED 5-FOLD STRATIFIED CV")
        self.logger.info("=" * 70)

        builder = EventBasedKFoldBuilder(self.df_manifest, self.logger)
        folds = builder.partition()

        output_base = os.path.join(self.output_dir, 'event_kfold')
        os.makedirs(output_base, exist_ok=True)

        for fold in folds:
            fold_dir = os.path.join(output_base, f"fold_{fold['fold_idx']}")
            self._build_fold_directory(fold, fold_dir)

        self.logger.info(f" Event-Based K-Fold complete: {len(folds)} folds")

    def build_spatial_lodo(self):
        self.logger.info("\n" + "=" * 70)
        self.logger.info("STRATEGY 1: SPATIAL LODO (Division-Level, 8 Folds)")
        self.logger.info("=" * 70)

        builder = SpatialLODOBuilder(self.df_manifest, self.logger)
        folds = builder.partition()

        output_base = os.path.join(self.output_dir, 'spatial_lodo')
        os.makedirs(output_base, exist_ok=True)

        for fold in folds:
            division_name = fold['division'].replace(' ', '_').replace('/', '_')
            fold_dir = os.path.join(output_base, f"lodo_division_{division_name}")
            self._build_fold_directory(fold, fold_dir)

        self.logger.info(f" Spatial LODO complete: {len(folds)} folds")

    def build_temporal_split(self):
        self.logger.info("\n" + "=" * 70)
        self.logger.info("STRATEGY 2: TEMPORAL SPLIT (Season-Adaptive Boundaries)")
        self.logger.info("=" * 70)

        builder = TemporalSplitBuilder(self.df_manifest, self.logger)
        split = builder.partition()

        output_dir = os.path.join(self.output_dir, 'temporal_split')
        self._build_fold_directory(split, output_dir)

        self.logger.info(f" Temporal Split complete")

    def build_spatio_temporal(self):
        self.logger.info("\n" + "=" * 70)
        self.logger.info("STRATEGY 3: COMBINED SPATIO-TEMPORAL (Division × Season × Era)")
        self.logger.info("=" * 70)

        builder = SpatioTemporalBuilder(self.df_manifest, self.logger)
        folds = builder.partition()

        output_base = os.path.join(self.output_dir, 'spatio_temporal')
        os.makedirs(output_base, exist_ok=True)

        for fold in folds:
            division_name = fold['division'].replace(' ', '_').replace('/', '_')
            season = fold['season']
            fold_dir = os.path.join(output_base, f"st_{division_name}_{season}")
            self._build_fold_directory(fold, fold_dir)

        self.logger.info(f" Spatio-Temporal complete: {len(folds)} folds")

    def build_all(self):
        self.logger.info("=" * 70)
        self.logger.info("HAZARDNET UNIFIED EXPERIMENTAL DATASET BUILDER")
        self.logger.info("(Event K-Fold + Division LODO + Season-Adaptive Temporal + Spatio-Temporal)")
        self.logger.info("=" * 70)
        self.logger.info(f"Total events in manifest: {len(self.df_manifest)}")

        # 1. BUILD MASTER HDF5 FILE ONCE
        master_h5_path = os.path.join(self.output_dir, 'master_tensors.h5')
        master_builder = MasterHDF5Serializer(self.tensor_dir, master_h5_path, self.logger)
        valid_ids = master_builder.build_master_hdf5(self.df_manifest)

        self.df_manifest = self.df_manifest[self.df_manifest['event_id'].astype(str).isin(valid_ids)].copy()
        self.logger.info(f"Active events after validation: {len(self.df_manifest)}")

        # 2. GENERATE ALL STRATEGY FOLDS
        self.build_event_kfold()       #  NEW
        self.build_spatial_lodo()
        self.build_temporal_split()
        self.build_spatio_temporal()

        # Save configuration
        config = {
            'master_h5_path': master_h5_path,
            'n_classes': HazardEncoder.n_classes(),
            'hazard_types': HazardEncoder.HAZARD_TYPES,
            'class_weights': {str(k): v for k, v in self.class_weights.items()},
            'seed': DataConfig.SEED,
            'n_folds_event_kfold': DataConfig.N_FOLDS,
            'target_tensor_shape': DataConfig.TARGET_TENSOR_SHAPE,
            'cropping_seasons': DataConfig.CROPPING_SEASONS,
            'season_climate_signatures': DataConfig.SEASON_CLIMATE_SIGNATURES,
            'season_temporal_splits': DataConfig.SEASON_TEMPORAL_SPLITS,
            'spatial_unit': 'division',
            'num_divisions': len(set(DataConfig.DISTRICT_TO_DIVISION.values())),
            'min_test_events': DataConfig.MIN_TEST_EVENTS,
            'strategies': ['event_kfold', 'spatial_lodo', 'temporal_split', 'spatio_temporal'],
            'note': 'Master HDF5 + CSV manifests. 4 validation strategies.'
        }

        config_path = os.path.join(self.output_dir, 'dataset_config.json')
        with open(config_path, 'w') as f:
            json.dump(config, f, indent=2)

        self.logger.info(f"\n Configuration saved to: {config_path}")
        self.logger.info("\n" + "=" * 70)
        self.logger.info("ALL EXPERIMENTAL VALIDATION STRATEGIES COMPLETE")
        self.logger.info("=" * 70)

    def summary(self):
        self.logger.info("\n" + "=" * 70)
        self.logger.info("UNIFIED EXPERIMENTAL DATASET BUILDER SUMMARY")
        self.logger.info("=" * 70)
        self.logger.info(f"Total events: {len(self.df_manifest)}")
        self.logger.info(f"N classes: {HazardEncoder.n_classes()}")
        self.logger.info(f"Target Tensor Shape: {DataConfig.TARGET_TENSOR_SHAPE}")
        self.logger.info(f"Spatial Unit: Division ({len(set(DataConfig.DISTRICT_TO_DIVISION.values()))} divisions)")
        self.logger.info(f"Cropping Seasons: Kharif-I (Mar-May), Kharif-II (Jun-Oct), Rabi (Nov-Feb)")
        self.logger.info(f"Strategies: event_kfold, spatial_lodo, temporal_split, spatio_temporal")
        for season, bounds in DataConfig.SEASON_TEMPORAL_SPLITS.items():
            self.logger.info(f"  {season}: Train 2000-{bounds['train_end']} | "
                             f"Val {bounds['train_end']+1}-{bounds['val_end']} | "
                             f"Test {bounds['val_end']+1}-2025")


# ============================================================================
# PYTORCH DATASET CLASS (Worker-Safe, Augmentation-Capable)
# ============================================================================

class MasterHDF5Dataset(Dataset):
    """
    Worker-safe, augmentation-capable PyTorch Dataset reading from Master HDF5.

    Features:
    - Worker-safe file handle via get_worker_info()
    - Returns 5 values: (tensor, label, severity, confidence, event_id)
    - Causality-preserving augmentation (brightness, contrast, temporal shift)
    - Nearest-neighbor spatial resize to prevent mask bleeding
    """

    def __init__(self, csv_path: str, master_h5_path: str, augment: bool = False):
        self.df = pd.read_csv(csv_path)
        self.master_h5_path = master_h5_path
        self.augment = augment
        self.h5f = None
        self._worker_id = None

        self.brightness = DataConfig.AUGMENT_PARAMS['brightness']
        self.contrast = DataConfig.AUGMENT_PARAMS['contrast']
        self.temporal_shift = DataConfig.AUGMENT_PARAMS['temporal_shift']
        self.target_shape = DataConfig.TARGET_TENSOR_SHAPE

    def _open_h5(self):
        worker_info = get_worker_info()
        current_worker_id = worker_info.id if worker_info is not None else -1

        if self.h5f is None or self._worker_id != current_worker_id:
            if self.h5f is not None:
                self.h5f.close()
            self.h5f = h5py.File(self.master_h5_path, 'r', rdcc_nbytes=1024**2*10)
            self._worker_id = current_worker_id

    def __len__(self):
        return len(self.df)

    def _resize_spatial(self, tensor: torch.Tensor) -> torch.Tensor:
        c, t, h, w = tensor.shape
        target_h, target_w = self.target_shape[2], self.target_shape[3]
        if h == target_h and w == target_w:
            return tensor
        tensor_reshaped = tensor.permute(1, 0, 2, 3).reshape(t * c, 1, h, w).contiguous()
        resized = F.interpolate(tensor_reshaped, size=(target_h, target_w), mode='nearest')
        return resized.reshape(t, c, target_h, target_w).permute(1, 0, 2, 3).contiguous()

    def _augment(self, tensor: torch.Tensor) -> torch.Tensor:
        if np.random.rand() > 0.5:
            shift = np.random.uniform(-self.brightness, self.brightness)
            tensor = tensor + shift

        if np.random.rand() > 0.5:
            factor = 1.0 + np.random.uniform(-self.contrast, self.contrast)
            mean = tensor.mean(dim=[-1, -2], keepdim=True)
            tensor = (tensor - mean) * factor + mean

        if np.random.rand() > 0.5:
            shift = np.random.randint(-self.temporal_shift, self.temporal_shift + 1)
            if shift > 0:
                boundary = tensor[:, 0:1, :, :].repeat(1, shift, 1, 1)
                tensor = torch.cat([boundary, tensor[:, :-shift, :, :]], dim=1)
            elif shift < 0:
                abs_shift = abs(shift)
                boundary = tensor[:, -1:, :, :].repeat(1, abs_shift, 1, 1)
                tensor = torch.cat([tensor[:, abs_shift:, :, :], boundary], dim=1)

        return tensor

    def __getitem__(self, idx):
        self._open_h5()

        row = self.df.iloc[idx]
        event_id = str(row['event_id'])

        tensor = torch.from_numpy(self.h5f['tensors'][event_id][:]).float()
        label = int(row['hazard_idx'])
        severity = float(row.get('severity_index', 0.0))
        confidence = float(row.get('confidence', 0.5))

        tensor = self._resize_spatial(tensor)

        if self.augment:
            tensor = self._augment(tensor)

        return tensor, label, severity, confidence, event_id

    def __del__(self):
        if self.h5f is not None:
            self.h5f.close()


# ============================================================================
# MAIN EXECUTION
# ============================================================================

if __name__ == '__main__':
    MANIFEST_PATH = '/kaggle/working/tensors_output/tensor_manifest.csv'
    TENSOR_DIR = '/kaggle/working/tensors_output/tensors'
    MASTER_OUTPUT_DIR = os.path.join(OUTPUT_DIR, 'HazardNet_Event_Based_Datasets')

    builder = ExperimentalDatasetBuilder(MANIFEST_PATH, TENSOR_DIR, MASTER_OUTPUT_DIR)
    builder.build_all()
    builder.summary()

    print("\n All experimental datasets ready!")
    print(f"   • Master HDF5 File: {os.path.join(OUTPUT_DIR, 'master_tensors.h5')}")
    print("   • Event K-Fold: 5-fold stratified CV (IEEE TGRS Table II)")
    print("   • Spatial LODO: Division-level, 8 folds (IEEE TGRS Table III)")
    print("   • Temporal Split: Season-adaptive boundaries (IEEE TGRS Table IV)")
    print("   • Spatio-Temporal: Division × Season × Era, up to 24 folds (IEEE TGRS Table V)")
    print("\n Usage in training:")
    print("   dataset = MasterHDF5Dataset(csv_path, master_h5_path, augment=True)")
    print("   loader = DataLoader(dataset, batch_size=16, num_workers=2)")

[2026-07-25 05:24:40,137] INFO: Loaded manifest: 2931 events
[2026-07-25 05:24:40,172] INFO: ======================================================================
[2026-07-25 05:24:40,173] INFO: HAZARDNET UNIFIED EXPERIMENTAL DATASET BUILDER
[2026-07-25 05:24:40,173] INFO: (Event K-Fold + Division LODO + Season-Adaptive Temporal + Spatio-Temporal)
[2026-07-25 05:24:40,175] INFO: ======================================================================
[2026-07-25 05:24:40,176] INFO: Total events in manifest: 2931
[2026-07-25 05:24:40,178] INFO: Building Master HDF5 at tensors_output/HazardNet_Event_Based_Datasets/master_tensors.h5...
Building Master HDF5: 100%|██████████| 2931/2931 [02:13<00:00, 21.96it/s]
[2026-07-25 05:26:53,638] INFO: Master HDF5 built successfully: 2931 events serialized.
[2026-07-25 05:26:53,642] INFO: Active events after validation: 2931
[2026-07-25 05:26:53,643] INFO: 
[2026-07-25 05:26:53,644] INFO: STRATEGY 0: EVENT-BASED 5-FOLD STRATIFIED CV
[2026-07-25 05:26:5


 All experimental datasets ready!
   • Master HDF5 File: tensors_output/master_tensors.h5
   • Event K-Fold: 5-fold stratified CV (IEEE TGRS Table II)
   • Spatial LODO: Division-level, 8 folds (IEEE TGRS Table III)
   • Temporal Split: Season-adaptive boundaries (IEEE TGRS Table IV)
   • Spatio-Temporal: Division × Season × Era, up to 24 folds (IEEE TGRS Table V)

 Usage in training:
   dataset = MasterHDF5Dataset(csv_path, master_h5_path, augment=True)
   loader = DataLoader(dataset, batch_size=16, num_workers=2)
